# SCDSA: Spatially Consistent Diffusion Synthetic Acceleration 

![Fuel-water quarter-domain with reflecting and vacuum boundaries](images/fuel_water_quarter.png)

This tutorial compares ordinary, unaccelerated power iteration with the same two-group eigenvalue calculation using Spatially Consistent Diffusion Synthetic Acceleration (SCDSA).

## Why a fuel-water problem?

SCDSA accelerates power iteration, so it requires a multiplying system rather than a nonfissile graphite block. A heterogeneous fuel--water problem is a useful demonstration: fission neutrons are born in the fast group, water moderates them, and the slowly varying thermal eigenmode is well represented by a diffusion correction.

The model is a 14 cm by 14 cm quarter-domain. Fuel occupies the 10 cm by 10 cm region adjacent to the reflecting left and bottom boundaries; water fills the remaining top and right reflector. The two-group fuel and water cross sections come from `test/assets/xs/xs_fuel_g2.xs` and `test/assets/xs/xs_water_g2.xs` in the regression suite. The outer boundaries are vacuum.

In [ ]:
from pathlib import Path

from mpi4py import MPI
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.context import Finalize
from pyopensn.logvol import RPPLogicalVolume
from pyopensn.mesh import KBAGraphPartitioner, OrthogonalMeshGenerator
from pyopensn.solver import (
    DiscreteOrdinatesProblem,
    PowerIterationKEigenSolver,
    SCDSAAcceleration,
)
from pyopensn.xs import MultiGroupXS

comm = MPI.COMM_WORLD
rank = comm.rank
tutorial_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
repo_root = next(
    path for path in (tutorial_dir, *tutorial_dir.parents)
    if (path / "test/assets/xs").is_dir()
)

## Build the transport problem

The mesh uses a 2 by 2 KBA partition and the generated script should be run with four MPI processes. SCDSA currently requires all energy groups to be in one groupset. The groupset contains no WGDSA, TGDSA, or other DSA options. The transport problem is rebuilt for each solve so both cases begin from the same state.

In [ ]:
def load_xs(filename):
    xs = MultiGroupXS()
    xs.LoadFromOpenSn(str(repo_root / "test/assets/xs" / filename))
    return xs


def make_problem():
    nodes = [0.5 * i for i in range(29)]
    partitioner = KBAGraphPartitioner(
        nx=2, ny=2, xcuts=[7.0], ycuts=[7.0]
    )
    mesh = OrthogonalMeshGenerator(
        node_sets=[nodes, nodes], partitioner=partitioner
    ).Execute()
    mesh.SetOrthogonalBoundaries()
    mesh.SetUniformBlockID(0)
    fuel_region = RPPLogicalVolume(
        xmin=-1.0, xmax=10.0, ymin=-1.0, ymax=10.0, infz=True
    )
    mesh.SetBlockIDFromLogicalVolume(fuel_region, 1, True)

    quadrature = GLCProductQuadrature2DXY(
        n_polar=4, n_azimuthal=8, scattering_order=1
    )
    return DiscreteOrdinatesProblem(
        mesh=mesh,
        num_groups=2,
        groupsets=[
            {
                "groups_from_to": (0, 1),
                "angular_quadrature": quadrature,
                "inner_linear_method": "petsc_richardson",
                "l_max_its": 2,
                "l_abs_tol": 1.0e-10,
            }
        ],
        xs_map=[
            {"block_ids": [0], "xs": load_xs("xs_water_g2.xs")},
            {"block_ids": [1], "xs": load_xs("xs_fuel_g2.xs")},
        ],
        boundary_conditions=[
            {"name": "xmin", "type": "reflecting"},
            {"name": "ymin", "type": "reflecting"},
            {"name": "xmax", "type": "vacuum"},
            {"name": "ymax", "type": "vacuum"},
        ],
        options={
            "save_angular_flux": True,
            "verbose_inner_iterations": False,
            "verbose_outer_iterations": False,
        },
    )

## Compare power iteration and SCDSA

The baseline passes no acceleration object to the power-iteration solver. The second case changes only this setting by passing an `SCDSAAcceleration` object with a piecewise-linear discontinuous diffusion discretization. We record $k_{\mathrm{eff}}$, completed power iterations, transport sweeps, and the maximum wall time across ranks.

In [ ]:
def solve(use_scdsa):
    problem = make_problem()
    solver_options = {
        "problem": problem,
        "k_tol": 1.0e-8,
        "max_iters": 300,
    }
    if use_scdsa:
        solver_options["acceleration"] = SCDSAAcceleration(
            problem=problem,
            sdm="pwld",
            l_abs_tol=1.0e-10,
            max_iters=100,
            pi_max_its=50,
            pi_k_tol=1.0e-10,
        )

    solver = PowerIterationKEigenSolver(**solver_options)
    comm.Barrier()
    start = MPI.Wtime()
    solver.Initialize()
    solver.Execute()
    comm.Barrier()
    elapsed = comm.allreduce(MPI.Wtime() - start, op=MPI.MAX)
    return (
        problem,
        solver.GetEigenvalue(),
        solver.GetNumPowerIterations(),
        solver.GetNumSweeps(),
        elapsed,
    )


_, unaccelerated_k, unaccelerated_iterations, unaccelerated_sweeps, unaccelerated_time = (
    solve(False)
)
scdsa_problem, scdsa_k, scdsa_iterations, scdsa_sweeps, scdsa_time = (
    solve(True)
)

## Interpret the metrics

Agreement in $k_{\mathrm{eff}}$ is the correctness check. Power iterations and transport sweeps measure convergence work; lower values indicate that SCDSA is removing the slowly converging eigenmode. Wall time is included for context but is not regression-tested because it depends on the machine.

In [ ]:
k_difference = abs(scdsa_k - unaccelerated_k)
speedup = unaccelerated_time / scdsa_time

if rank == 0:
    print(f"Unaccelerated k-effective={unaccelerated_k:.12e}")
    print(f"SCDSA k-effective={scdsa_k:.12e}")
    print(f"SCDSA k-effective difference={k_difference:.12e}")
    print(f"Unaccelerated power iteration count={unaccelerated_iterations}")
    print(f"SCDSA power iteration count={scdsa_iterations}")
    print(f"Unaccelerated sweeps={unaccelerated_sweeps}")
    print(f"SCDSA sweeps={scdsa_sweeps}")
    print(f"Unaccelerated wall time (s)={unaccelerated_time:.6f}")
    print(f"SCDSA wall time (s)={scdsa_time:.6f}")
    print(f"SCDSA speedup={speedup:.6f}")

assert k_difference < 1.0e-6
assert scdsa_iterations < unaccelerated_iterations
assert scdsa_sweeps < unaccelerated_sweeps

A representative four-process run gives the following eigenvalue comparison:

| Solve | $k_{\mathrm{eff}}$ |
|---|---:|
| No acceleration | 0.59638190 |
| SCDSA | 0.59638194 |

The same run gives the convergence and timing comparison:

| Solve | Power iterations | Transport sweeps | Wall time (s) |
|---|---:|---:|---:|
| No acceleration | 96 | 480 | 0.358 |
| SCDSA | 16 | 80 | 0.318 |

The eigenvalues differ by about $3.8\times10^{-8}$, below the $10^{-6}$ comparison tolerance. SCDSA uses about 83% fewer power iterations and transport sweeps and gives a representative speedup of 1.13. The iteration counts are deterministic for this configuration, while wall time is machine-dependent.

## Export and visualize the scalar flux

The converged eigenvalues agree, so the SCDSA solution is used as the representative flux field. The following code exports both group scalar fluxes to VTK. It remains in Markdown so regression tests do not create output files. Select the thermal-group field in ParaView when creating the flux-profile image.

```python
from pyopensn.fieldfunc import FieldFunctionGridBased

fast_flux, thermal_flux = scdsa_problem.GetScalarFluxFieldFunction()
FieldFunctionGridBased.ExportMultipleToPVTU(
    [fast_flux, thermal_flux], "Flux/SCDSA_Flux"
)
```

![Flux Profile g0](images/SCDSA_g0.png)
![Flux Profile g1](images/SCDSA_g1.png)



## Finalize (for Jupyter Notebook only)

In script mode, PyOpenSn handles finalization automatically. In a Jupyter kernel, finalize OpenSn before MPI.

In [ ]:
if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()